In [0]:
%run ./utils

# Imports

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load

In [0]:
dbutils.widgets.text("silver_table_name", "tb_atendentes")
silver_table_name = dbutils.widgets.get("silver_table_name")

df_atendente = load_df(silver_table_name)


# Atributos


In [0]:
# surrogate key (big int)
df_dim_atendente = add_sequential_key(df_atendente, "atendente_sk")

In [0]:
# As datas já vêm tipadas da silver; aqui só o nível vira int
df_dim_atendente = (
    df_dim_atendente
        .withColumn("nivel_atendimento", F.col("nivel_atendimento").cast("int"))
        .select(
            "atendente_sk",
            "id_atendente",
            "nome_atendente",
            "nivel_atendimento",
            "data_admissao",
            "data_desligamento",
            "status_atendente"
        )
)

display(df_dim_atendente)

In [ ]:
# Membro desconhecido: a fato usa atendente_sk = -1 quando o chamado não tem atendente na dimensão
df_unknown = spark.range(1).select(
    F.lit(-1).cast("bigint").alias("atendente_sk"),
    F.lit(None).cast("string").alias("id_atendente"),
    F.lit("NÃO INFORMADO").alias("nome_atendente"),
    F.lit(None).cast("int").alias("nivel_atendimento"),
    F.lit(None).cast("date").alias("data_admissao"),
    F.lit(None).cast("date").alias("data_desligamento"),
    F.lit("NÃO INFORMADO").alias("status_atendente"),
)

df_dim_atendente = df_dim_atendente.unionByName(df_unknown)


# Verificação

In [0]:
# dimensão deve ter uma linha por atendente (SCD tipo 1) mais o membro desconhecido, e a chave deve ser única
total = df_dim_atendente.count()
atendente = df_dim_atendente.filter(F.col("atendente_sk") != -1).select("id_atendente").distinct().count()
chaves = df_dim_atendente.select("atendente_sk").distinct().count()
print(f"Linhas: {total} (inclui o membro -1) | id_atendente únicos: {atendente} | atendente_sk únicas: {chaves}")

# a dimensão tem as linhas da silver mais o membro desconhecido
total_silver = df_atendente.count()
print(f"Linhas na Silver: {total_silver} | Linhas na dimensão: {total} (silver + membro -1)")

# nível deve ser 1 ou 2 (o cast para int anula valores inválidos)
display(df_dim_atendente.groupBy("nivel_atendimento").count())


In [0]:
dbutils.widgets.text("gold_table_name", "dim_atendente")
gold_table_name = dbutils.widgets.get("gold_table_name")

save_df(df_dim_atendente, gold_table_name)